# 01 — Data Preprocessing

**Project:** Predictive Analytics for Enterprise Streaming Acquisitions  
**Course:** CMPS 451 — Data Mining, Big Data & Analytics (Spring 2026)  
**Team:** 11

---

## Objectives
1. Load all 7 IMDb TSV datasets using PySpark
2. Clean and filter data (remove adult content, irrelevant title types, missing values)
3. Filter unreliable ratings (numVotes < 100) per instructor feedback
4. Join all tables into a unified dataset
5. Integrate the IEEE DataPort User Ratings dataset (Dataset.npy)
6. Save the preprocessed data for downstream analysis

In [ ]:

# ── Imports & Environment Setup ──
import os
import sys
import subprocess
import numpy as np
import pandas as pd
import findspark

os.environ['HADOOP_HOME'] = r'C:\hadoop'
os.environ['SPARK_HOME'] = r'C:\spark'
os.environ['PATH'] = r'C:\hadoop\bin;' + os.environ.get('PATH', '')
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

findspark.init()

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, FloatType
)

# Paths
DATA_DIR = os.path.join("Dataset")
OUTPUT_DIR = os.path.join("outputs")
os.makedirs(os.path.join(OUTPUT_DIR, "parquet"), exist_ok=True)

print("Setup complete.")

Exception: Unable to find py4j in A:\spark\python, your SPARK_HOME may not be configured correctly

In [ ]:
# ── Initialize Spark (pseudo-distributed mode) ──
spark = (
    SparkSession.builder
    .appName("IMDb_Preprocessing")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.ui.showConsoleProgress", "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")
print(f"Running in pseudo-distributed mode with {spark.sparkContext.defaultParallelism} cores")

## 1. Load IMDb Datasets

In [ ]:
def load_tsv(filename):
    """Load a gzipped TSV file into a Spark DataFrame."""
    path = os.path.join(DATA_DIR, filename)
    df = (
        spark.read
        .option("header", "true")
        .option("sep", "\t")
        .option("nullValue", "\\N")
        .option("quote", "")
        .csv(path)
    )
    print(f"Loaded {filename}: {df.count():,} rows, {len(df.columns)} columns")
    return df

# Load all tables
df_basics    = load_tsv("title.basics.tsv.gz")
df_ratings   = load_tsv("title.ratings.tsv.gz")
df_crew      = load_tsv("title.crew.tsv.gz")
df_akas      = load_tsv("title.akas.tsv.gz")
df_principals= load_tsv("title.principals.tsv.gz")
df_episodes  = load_tsv("title.episode.tsv.gz")
df_names     = load_tsv("name.basics.tsv.gz")

In [ ]:
# ── Inspect schemas ──
print("=== title.basics ===")
df_basics.printSchema()
df_basics.show(3, truncate=False)

print("\n=== title.ratings ===")
df_ratings.printSchema()
df_ratings.show(3, truncate=False)

## 2. Data Cleaning & Filtering

In [ ]:
# ── Step 2a: Filter by title type ──
VALID_TYPES = ["movie", "tvSeries", "tvMovie", "tvMiniSeries"]
df_basics_filtered = df_basics.filter(F.col("titleType").isin(VALID_TYPES))

print("Title type distribution (after filter):")
df_basics_filtered.groupBy("titleType").count().orderBy("count", ascending=False).show()

In [ ]:
# ── Step 2b: Remove adult content ──
df_basics_filtered = df_basics_filtered.filter(F.col("isAdult") == "0")
print(f"After removing adult content: {df_basics_filtered.count():,} titles")

In [ ]:
# ── Step 2c: Cast numeric columns ──
df_basics_filtered = (
    df_basics_filtered
    .withColumn("startYear", F.col("startYear").cast(IntegerType()))
    .withColumn("endYear", F.col("endYear").cast(IntegerType()))
    .withColumn("runtimeMinutes", F.col("runtimeMinutes").cast(IntegerType()))
)

df_ratings = (
    df_ratings
    .withColumn("averageRating", F.col("averageRating").cast(FloatType()))
    .withColumn("numVotes", F.col("numVotes").cast(IntegerType()))
)

print("Numeric columns cast successfully.")

In [ ]:
# ── Step 2d: Drop rows with critical missing values ──
before = df_basics_filtered.count()
df_basics_clean = df_basics_filtered.dropna(subset=["startYear", "runtimeMinutes", "genres"])
after = df_basics_clean.count()
print(f"Dropped {before - after:,} rows with missing startYear/runtimeMinutes/genres")
print(f"Remaining: {after:,} titles")

## 3. Join Tables

In [ ]:
# ── Step 3a: Join basics with ratings ──
df_main = df_basics_clean.join(df_ratings, on="tconst", how="inner")
print(f"After joining with ratings: {df_main.count():,} titles")
df_main.show(3, truncate=False)

In [ ]:
# ── Step 3b: Filter by numVotes (instructor feedback) ──
MIN_VOTES = 100
before = df_main.count()
df_main = df_main.filter(F.col("numVotes") >= MIN_VOTES)
after = df_main.count()
print(f"Filtered titles with numVotes < {MIN_VOTES}")
print(f"Removed: {before - after:,} titles")
print(f"Remaining: {after:,} titles with reliable ratings")

In [ ]:
# ── Step 3c: Join with crew (directors and writers) ──
df_main = df_main.join(df_crew, on="tconst", how="left")
print(f"After joining with crew: {df_main.count():,} titles")

In [ ]:
# ── Step 3d: Aggregate language info from title.akas ──
df_lang = (
    df_akas
    .filter(F.col("language").isNotNull())
    .groupBy("titleId", "language")
    .count()
    .withColumn(
        "rank",
        F.row_number().over(
            Window.partitionBy("titleId").orderBy(F.desc("count"))
        )
    )
    .filter(F.col("rank") == 1)
    .select(
        F.col("titleId").alias("tconst"),
        F.col("language").alias("primaryLanguage")
    )
)

df_region_count = (
    df_akas
    .filter(F.col("region").isNotNull())
    .groupBy("titleId")
    .agg(F.countDistinct("region").alias("numRegions"))
    .withColumnRenamed("titleId", "tconst")
)

df_main = df_main.join(df_lang, on="tconst", how="left")
df_main = df_main.join(df_region_count, on="tconst", how="left")
print(f"After joining with language/region info: {df_main.count():,} titles")

In [ ]:
# ── Step 3e: Aggregate principal cast/crew info ──
df_principals_agg = (
    df_principals
    .groupBy("tconst")
    .agg(
        F.count("nconst").alias("numPrincipals"),
        F.countDistinct("category").alias("numRoleTypes")
    )
)
df_main = df_main.join(df_principals_agg, on="tconst", how="left")
print(f"After joining with principals: {df_main.count():,} titles")

## 4. Integrate User Ratings (Dataset.npy)

User ratings aggregation is done in pandas because PySpark's CSV reader on Windows has issues with paths containing spaces. The aggregated result is merged with the main dataset via pandas.

In [ ]:
# ── Load and parse user ratings ──
print("Loading Dataset.npy (4.67M user ratings)...")
user_ratings_raw = np.load(os.path.join(DATA_DIR, "Dataset.npy"), allow_pickle=True)
print(f"Loaded {len(user_ratings_raw):,} user ratings")
print(f"Sample: {user_ratings_raw[:3]}")

parsed = []
for row in user_ratings_raw:
    parts = row.split(",")
    if len(parts) >= 3:
        parsed.append((parts[0].strip(), parts[1].strip(), int(parts[2].strip())))

pdf_user = pd.DataFrame(parsed, columns=["userId", "titleId", "userRating"])
print(f"\nParsed {len(pdf_user):,} ratings")
print(f"Unique users: {pdf_user['userId'].nunique():,}")
print(f"Unique titles: {pdf_user['titleId'].nunique():,}")
pdf_user.head()

In [ ]:
# ── Aggregate user-level features per title (in pandas) ──
print("Aggregating user ratings per title...")
df_user_agg_pd = (
    pdf_user
    .groupby("titleId")
    .agg(
        userRatingCount=("userRating", "count"),
        userRatingMean=("userRating", "mean"),
        userRatingStd=("userRating", "std"),
        userRatingMin=("userRating", "min"),
        userRatingMax=("userRating", "max"),
    )
    .reset_index()
)

# Extreme rating count (ratings <= 2 or >= 9)
extreme_counts = (
    pdf_user
    .assign(is_extreme=lambda x: (x['userRating'] <= 2) | (x['userRating'] >= 9))
    .groupby("titleId")["is_extreme"]
    .sum()
    .reset_index()
    .rename(columns={"is_extreme": "extremeRatingCount"})
)

df_user_agg_pd = df_user_agg_pd.merge(extreme_counts, on="titleId")
df_user_agg_pd["extremeRatingRatio"] = df_user_agg_pd["extremeRatingCount"] / df_user_agg_pd["userRatingCount"]
df_user_agg_pd["userRatingRange"] = df_user_agg_pd["userRatingMax"] - df_user_agg_pd["userRatingMin"]
df_user_agg_pd.rename(columns={"titleId": "tconst"}, inplace=True)

print(f"Aggregated user features for {len(df_user_agg_pd):,} unique titles")
df_user_agg_pd.head()

In [ ]:
# ── Merge user features with main dataset via pandas ──
print("Converting main Spark DataFrame to pandas for merge...")
pdf_main = df_main.toPandas()
print(f"Main dataset: {len(pdf_main):,} titles")

# Merge
pdf_main = pdf_main.merge(df_user_agg_pd, on="tconst", how="left")

# Fill nulls for titles without user ratings
user_cols = ["userRatingCount", "userRatingMean", "userRatingStd",
             "userRatingMin", "userRatingMax", "extremeRatingCount",
             "extremeRatingRatio", "userRatingRange"]
pdf_main[user_cols] = pdf_main[user_cols].fillna(0)

matched = (pdf_main['userRatingCount'] > 0).sum()
print(f"\nTitles WITH user ratings: {matched:,} / {len(pdf_main):,}")
print(f"Columns: {len(pdf_main.columns)}")
pdf_main[['tconst', 'primaryTitle', 'averageRating', 'userRatingCount', 'userRatingMean']].head(10)

## 5. Summary & Save

In [ ]:
# ── Summary statistics ──
print("=== Dataset Shape ===")
print(f"Rows: {len(pdf_main):,}")
print(f"Columns: {len(pdf_main.columns)}")

print("\n=== Numeric Summary ===")
print(pdf_main[['averageRating', 'numVotes', 'runtimeMinutes', 'startYear',
                'numRegions', 'numPrincipals', 'userRatingCount', 'userRatingMean']].describe())

print("\n=== Title Type Distribution ===")
print(pdf_main['titleType'].value_counts())

print("\n=== Language Distribution (Top 15) ===")
print(pdf_main['primaryLanguage'].value_counts().head(15))

In [ ]:
# ── Save preprocessed data as CSV ──
output_path = os.path.join(OUTPUT_DIR, "parquet", "preprocessed")
os.makedirs(output_path, exist_ok=True)
save_path = os.path.join(output_path, "data.csv")
pdf_main.to_csv(save_path, index=False)

print(f"Preprocessed data saved to: {save_path}")
print(f"Total rows: {len(pdf_main):,}")
print(f"Total columns: {len(pdf_main.columns)}")


In [ ]:
# ── Verify saved data ──
df_check = pd.read_csv(save_path)
print(f"Verification - loaded {len(df_check):,} rows")
print(f"User ratings check: max userRatingCount = {df_check['userRatingCount'].max()}")

In [ ]:
print("Preprocessing complete!")
print("Proceed to 02_feature_engineering.ipynb") 